# Phase 6 — Translation Agent

**Plan ref**: §6 Stage 6

Tier-aware translation pipeline:

| Tier | Pipeline | Output |
|------|---------|--------|
| primary | word.py → paragraph.py → review.py | `textCanonical`, `textVernacular`, `textVernacularJa` (if ja/kanbun) |
| secondary | normalize → concept extraction only | `textCanonical`, `semanticConcepts` |

Sections:
1. Setup + environment
2. Seed Dictionary KB + Norms KB
3. Tokenizer smoke test (jieba / fugashi)
4. Normalization pipeline smoke test
5. Word-analysis smoke test on 3 primary chunks
6. Full primary-tier translation on 5 chunks
7. Secondary-tier concept extraction on 5 chunks
8. Batch translate (smoke: N=20 chunks)
9. Quality check + distribution
10. Write artifact

In [1]:
import sys, json, logging
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from dotenv import load_dotenv
load_dotenv()

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s %(message)s')
logging.getLogger('neo4j.notifications').setLevel(logging.WARNING)
logging.getLogger('httpx').setLevel(logging.WARNING)

ARTIFACT_DIR = Path('_artifacts/06_translation')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Smoke-mode settings
N_WORD_SMOKE = 3     # chunks to run word analysis on
N_TRANSLATE  = 5     # chunks for primary-tier full translation
N_SECONDARY  = 5     # chunks for secondary-tier concept extraction
N_BATCH      = 20    # batch run size

print('Setup complete')

Setup complete


In [2]:
from apps.backend.graph.neo4j_client import get_driver
from apps.backend.llm.silra import get_silra_client
import os

driver = get_driver()
client = get_silra_client()
CHAT_MODEL = os.getenv('CHAT_LLM_MODEL', 'deepseek-chat')

print(f'Neo4j connected, chat model: {CHAT_MODEL}')

Neo4j connected, chat model: deepseek-chat


## 1. Seed Dictionary KB + Norms KB

In [3]:
from apps.backend.kb.dictionary import seed_dictionary
from apps.backend.kb.norms import seed_norms

n_dict = seed_dictionary(driver)
n_norms = seed_norms(driver)
print(f'Dictionary entries seeded: {n_dict}')
print(f'Norms seeded:              {n_norms}')

# Verify in Neo4j
with driver.session() as s:
    dict_count = s.run('MATCH (d:DICTIONARY_ENTRY) RETURN count(d) AS n').single()['n']
    norm_count  = s.run('MATCH (n:NORM) RETURN count(n) AS n').single()['n']
print(f'Total DICTIONARY_ENTRY: {dict_count}, Total NORM: {norm_count}')

INFO apps.backend.kb.dictionary seed_dictionary: upserted 53 entries from /Users/mohasani/Ancient/data/seeds/dictionary_seed.jsonl
INFO apps.backend.kb.norms seed_norms: upserted 31 norms from /Users/mohasani/Ancient/data/seeds/norms_seed.jsonl


Dictionary entries seeded: 53
Norms seeded:              31
Total DICTIONARY_ENTRY: 53, Total NORM: 31


## 2. Tokenizer Smoke Test

In [4]:
from apps.backend.lang.tokenizer import tokenize

test_cases = [
    ('貞觀十九年，太宗皇帝詔令群臣商議府兵制度之改革', 'zh-classical'),
    ('諸謀反及大逆者，皆斬', 'zh-classical'),
    ('律疏議第一卷總則', 'zh-classical'),
]

for text, lang in test_cases:
    tokens = tokenize(text, lang)
    surfs = [t.surface for t in tokens]
    print(f'  [{lang}] "{text[:30]}..."')
    print(f'   → {surfs[:10]}')

Building prefix dict from the default dictionary ...
DEBUG jieba Building prefix dict from the default dictionary ...
Loading model from cache /var/folders/d8/1fbq_k6x2n7b78_4g7xchblr0000gn/T/jieba.cache
DEBUG jieba Loading model from cache /var/folders/d8/1fbq_k6x2n7b78_4g7xchblr0000gn/T/jieba.cache
Loading model cost 0.283 seconds.
DEBUG jieba Loading model cost 0.283 seconds.
Prefix dict has been built successfully.
DEBUG jieba Prefix dict has been built successfully.


  [zh-classical] "貞觀十九年，太宗皇帝詔令群臣商議府兵制度之改革..."
   → ['貞觀', '十九年', '，', '太宗皇帝', '詔令', '群臣', '商議', '府兵制', '度', '之']
  [zh-classical] "諸謀反及大逆者，皆斬..."
   → ['諸謀', '反及', '大', '逆者', '，', '皆', '斬']
  [zh-classical] "律疏議第一卷總則..."
   → ['律', '疏議', '第一卷', '總則']


## 3. Normalization Pipeline Smoke Test

In [5]:
from apps.backend.normalize.pipeline import normalize_canonical

norm_cases = [
    ('贞观十九年诏令', 'zh', 'Tang'),
    ('諸謀反及大逆者', 'zh', 'Tang'),
]

for text, lang, era in norm_cases:
    result = normalize_canonical(text, lang=lang, era=era, apply_loan=True)
    print(f'  input:     {text!r}')
    print(f'  canonical: {result.canonical!r}')
    print()

TypeError: normalize_canonical() got an unexpected keyword argument 'loan'

## 4. Dictionary Lookup Smoke Test

In [ ]:
from apps.backend.kb.dictionary import lookup_term

terms = ['令', '律', '謀反', '三省', '吏部', '貞觀']
print('Dictionary lookups:')
for term in terms:
    entries = lookup_term(driver, term, 'zh')
    if entries:
        print(f'  {term}: {entries[0].meaning[:60]}')
    else:
        print(f'  {term}: (not found)')

## 5. Word Analysis Smoke Test

In [ ]:
from apps.backend.agents.translation.word import analyze_words

# Sample primary-tier chunks from Neo4j
with driver.session() as s:
    primary_chunks = s.run(
        "MATCH (ch:CHUNK)-[:HAS]-(p:PAGE) "
        "WHERE p.tier = 'primary' AND ch.text IS NOT NULL AND size(ch.text) > 50 "
        "AND (ch.translationStatus IS NULL OR ch.translationStatus = 'failed') "
        "RETURN ch.id AS id, ch.text AS text, p.language AS lang, "
        "p.detectedEra AS era, p.tier AS tier "
        "LIMIT $n",
        n=N_WORD_SMOKE
    ).data()

print(f'Sampled {len(primary_chunks)} primary chunks for word analysis')
word_results = []
for row in primary_chunks:
    print(f'\nAnalyzing chunk {row["id"][:50]}...')
    wr = analyze_words(
        row['text'],
        row.get('lang') or 'zh-classical',
        row.get('era'),
        row.get('tier') or 'primary',
        driver,
        client=client,
        model=CHAT_MODEL,
    )
    word_results.append((row, wr))
    print(f'  canonical:  {wr.text_canonical[:80]}...')
    print(f'  tokens:     {len(wr.tokens)}')
    polysemy = sum(1 for t in wr.tokens if t.polysemy_resolved)
    with_def = sum(1 for t in wr.tokens if t.definition)
    print(f'  with_def:   {with_def}, polysemy_resolved: {polysemy}')

## 6. Full Primary-Tier Translation

In [ ]:
from apps.backend.agents.translation.paragraph import translate_paragraph
from apps.backend.agents.translation.review import review_translation

with driver.session() as s:
    chunks_primary = s.run(
        "MATCH (ch:CHUNK)-[:HAS]-(p:PAGE) "
        "WHERE p.tier = 'primary' AND ch.text IS NOT NULL AND size(ch.text) > 50 "
        "AND (ch.translationStatus IS NULL OR ch.translationStatus = 'failed') "
        "RETURN ch.id AS id, ch.text AS text, p.language AS lang, "
        "p.detectedEra AS era "
        "LIMIT $n",
        n=N_TRANSLATE
    ).data()

full_translations = []
for row in chunks_primary:
    lang = row.get('lang') or 'zh-classical'
    era  = row.get('era')
    print(f'\nTranslating chunk {row["id"][:50]}...')

    wr = analyze_words(row['text'], lang, era, 'primary', driver, client=client, model=CHAT_MODEL)
    pr = translate_paragraph(wr, lang, driver, client=client, model=CHAT_MODEL)
    rr = review_translation(wr, pr, lang, client=client, model=CHAT_MODEL)

    full_translations.append({'id': row['id'], 'canonical': wr.text_canonical, 'vernacular': rr.text_final})

    print(f'  canonical:  {wr.text_canonical[:80]}...')
    print(f'  vernacular: {rr.text_final[:100]}...')
    print(f'  review: iterations={rr.iterations} all_ok={rr.all_ok}')

print(f'\nCompleted {len(full_translations)} full translations')

## 7. Secondary-Tier Concept Extraction

In [ ]:
import json as _json
from apps.backend.normalize.pipeline import normalize_canonical

with driver.session() as s:
    chunks_sec = s.run(
        "MATCH (ch:CHUNK)-[:HAS]-(p:PAGE) "
        "WHERE p.tier = 'secondary' AND ch.text IS NOT NULL AND size(ch.text) > 50 "
        "AND (ch.translationStatus IS NULL OR ch.translationStatus = 'failed') "
        "RETURN ch.id AS id, ch.text AS text, p.language AS lang, "
        "p.detectedEra AS era "
        "LIMIT $n",
        n=N_SECONDARY
    ).data()

print(f'Sampled {len(chunks_sec)} secondary-tier chunks')

# Secondary: normalize + concept extraction only
from apps.backend.pipeline.translate import _extract_concepts

for row in chunks_sec:
    lang = row.get('lang') or 'zh-modern'
    era  = row.get('era')
    norm = normalize_canonical(row['text'], lang='zh', era=era or '', apply_loan=False)
    concepts = _extract_concepts(norm.canonical, client, CHAT_MODEL)
    print(f'  {row["id"][:50]}: {concepts}')

## 8. Batch Translate (Smoke: N=20)

In [ ]:
from apps.backend.pipeline.translate import translate_chunks

report = translate_chunks(
    driver,
    limit=N_BATCH,
    client=client,
    model=CHAT_MODEL,
)

print(f'Batch translation:')
print(f'  total:   {report.total}')
print(f'  ok:      {report.ok}')
print(f'  failed:  {report.failed}')
print(f'  skipped: {report.skipped}')
print(f'  prompt_tokens: {report.prompt_tokens:,}')
print(f'  completion_tokens: {report.completion_tokens:,}')

## 9. Quality Check

In [ ]:
with driver.session() as s:
    status_dist = s.run(
        'MATCH (ch:CHUNK) WHERE ch.translationStatus IS NOT NULL '
        'RETURN ch.translationStatus AS status, count(*) AS n '
        'ORDER BY n DESC'
    ).data()
    sample_translated = s.run(
        "MATCH (ch:CHUNK) WHERE ch.translationStatus = 'ok' AND ch.textVernacular IS NOT NULL "
        "RETURN ch.id AS id, ch.text AS orig, ch.textCanonical AS canonical, "
        "ch.textVernacular AS vernacular "
        "LIMIT 3"
    ).data()

print('Translation status distribution:')
for row in status_dist:
    print(f'  {row["status"]:15s}: {row["n"]:,}')

print('\nSample translated chunks:')
for row in sample_translated:
    orig = (row.get('orig') or '')[:60]
    vern = (row.get('vernacular') or '')[:100]
    print(f'  ORIG:      {orig}')
    print(f'  VERNACULAR:{vern}')
    print()

## 10. Write Artifact

In [ ]:
from datetime import datetime, timezone

artifact = {
    'phase': '06_translation_agent',
    'ts': datetime.now(timezone.utc).isoformat(),
    'seeds': {
        'dictionary_entries': dict_count,
        'norms': norm_count,
    },
    'batch_report': report.to_dict(),
    'status_distribution': {row['status']: row['n'] for row in status_dist},
    'smoke_primary_translations': [
        {'id': t['id'][:50], 'canonical': t['canonical'][:80], 'vernacular': t['vernacular'][:100]}
        for t in full_translations
    ],
}

out = ARTIFACT_DIR / 'report.json'
out.write_text(json.dumps(artifact, ensure_ascii=False, indent=2))
print(f'Artifact written → {out}')
print(json.dumps({
    'total': report.total,
    'ok': report.ok,
    'failed': report.failed,
    'skipped': report.skipped,
    'dict_entries': dict_count,
    'norms': norm_count,
}, indent=2))